<a href="https://colab.research.google.com/github/Cooljoe67/ML_DSAI/blob/main/12_kelvin_nested_and_stability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nested Cross-Validation and Hyperparameter Stability Analysis
In this notebook, we’ll explore how to tune a model’s hyperparameters properly using something called nested cross-validation. This helps us avoid data leakage and gives us a better idea of how well our model settings hold up across different slices of data.

To keep things nice and simple, we’ll be working with a small dataset and using a Support Vector Classifier (SVC) as our model.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate


# set numpy random seed for consistent data creation
np.random.seed(42)

# Example data
X = np.random.rand(1000, 5)
y = np.random.randint(0, 2, size=1000)

In [ ]:
# Define the pipeline: scaler + SVC
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', gamma='scale'))
])

# Hyperparameter grid for GridSearchCV (we tune only 'C' here)
param_grid = {
    'svc__C': [0.01, 0.1, 1, 10, 100],

    }

## 2. Understanding CV vs. Nested CV 📘


### What is Cross-Validation (CV)?
Instead of a single train/test split, **k-fold cross-validation** provides a much more reliable estimate of how your model will perform on unseen data:
* **Split** the dataset into k equal parts (folds).
* **Train** the model on k-1 folds.
* **Test** on the 1 remaining fold.
* **Repeat** k times so every fold acts as the test set exactly once, then average the results.

### **The Problem:** Hyperparameter Overfitting
Models have settings (hyperparameters) we have to choose manually, like the `C` value in an `SVC`. If we repeatedly tweak these settings to maximize our standard CV score, the model becomes specifically tuned to those folds. The resulting score becomes overly optimistic and doesn't reflect true real-world performance.

### **The Solution:** Nested CV
Nested CV fixes this by adding a second layer of validation to ensure our final test is completely blind:
* **Inner Loop (Tuning):** Splits the training data into its own CV folds just to find the best hyperparameters.
* **Outer Loop (Evaluation):** Takes the model configured by the inner loop and tests it on the untouched outer fold.


> **The Golden Rule:** Never report the CV score you used for tuning as your final performance metric. Always use a nested approach (or a completely locked-away test set) to report how your model handles truly new data.

---
## 3.&nbsp;Implementing Nested Cross-Validation 🔁


### 3.1 Code for Nested Cross Validation

Here’s a breakdown of what the code below does. We’re using a Support Vector Classifier (`SVC`) with a pipeline that includes scaling, and we’re testing different values for the parameter **C** using nested cross validation:
1. **Outer Cross Validation**
   - The whole dataset is split into 5 folds.
   - One of those folds is kept aside for testing, and the remaining four are used for training. This is repeated so each fold gets used as the test set once.
2. **Inner Cross Validation**
   - Inside each training set from the outer loop, we do more splitting.
   - This is where we try out different values for **C** to figure out which one works best.
3. **Performance Check**
   - Once we’ve chosen the best **C** using the inner loop, we test the model on the outer fold’s test data.
   - This gives us a good idea of how well the model performs after tuning, without any bias from seeing the test data early.



In [ ]:
# Set up inner and outer cross-validation
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)
outer_cv = inner_cv

# Prepare the GridSearch for the inner loop
grid_search = GridSearchCV(pipe, param_grid, cv=inner_cv)

# Perform nested cross-validation
nested_score_results = cross_validate(grid_search, X, y, cv=outer_cv, return_estimator=True)


In [ ]:
# Gather results
outer_scores = nested_score_results['test_score']
print("Outer CV accuracy scores for each fold: ", outer_scores)
print(f"Mean accuracy (nested CV estimate): {outer_scores.mean():.3f} (std: {outer_scores.std():.3f})")

**Outer CV accuracy scores**: These are the accuracy results on the test sets from each outer fold. They show how well the model performed on data it hasn’t seen during training.

**Mean accuracy**: This gives us a reliable idea of how well our model (after we’ve picked the best hyperparameters) is likely to perform on new, unseen data.

**Best Parameter Per Outer Fold**: This gives us a reliable idea of how well our model (after we’ve picked the best hyperparameters) is likely to perform on new, unseen data.

In [ ]:
# how to find the first outer folds best params
nested_score_results['estimator']#[0].best_params_['svc__C']

In [ ]:
# Retrieve the best C values chosen in each outer fold
best_c_each_fold = []
for best in nested_score_results['estimator']:
    best_c_each_fold.append(best.best_params_['svc__C'])

print("Best C hyperparameter in each fold: ", best_c_each_fold)

### 3.2 Retraining the Final Model

Nested CV gives you an honest performance estimate, but it doesn't give you a final model. To deploy your model into the real world, you must retrain it on **100% of your dataset**.

**How do you choose the final hyperparameter?**
* **The Majority Vote (Most Common):** Pick the value that won most frequently across your outer loops (e.g., if `C=0.1` won 4 out of 5 times, use `C=0.1`).
* **The Average:** If you are tuning a continuous numerical parameter, you can average the winning values (rarely used for categorical parameters).

In [ ]:
from collections import Counter # useful import for counting strings, numbers, tuples

counts = Counter(best_c_each_fold)
final_c = counts.most_common(1)[0][0]  # This picks the most frequently chosen C
print(f"Most frequently chosen C: {final_c}")


Then retrain on the entire dataset if you chosen the most frequent C

In [ ]:
final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', gamma='scale', C=final_c))
])

final_model.fit(X, y)

Your `final_model` is now trained on all the data, using the hyperparameter that consistently performed best during the outer folds of your nested cross-validation. Since you're using the entire dataset for training, you won’t get a new validation score for this final model. But that’s alright — the average score from your nested cross-validation still gives you a good idea of how well the model is likely to perform on new data.

---
## 4.&nbsp;Hyperparameter Stability Analysis 📊

We picked a value for **C**, but was it a lucky choice? If we shuffled the data differently, would the model pick a completely different hyperparameter?


**Why Stability Matters:**
* **Sensitivity:** If the "best" parameter changes wildly with every random split, it means multiple settings perform identically, or your model is highly unstable.
* **Simplicity vs. Complexity:** If two parameters yield the exact same performance, stability analysis gives you the confidence to default to the simpler, heavily regularised model.
* **Reproducibility:** A stable hyperparameter choice proves your model isn't just relying on "lucky" data splits.

### 4.1 Check Performance Across Different C Values
We run cross-validation once on the whole dataset to check how the average accuracy changes as we try out different values of C. The `mean_scores` tells us the average accuracy across the folds, while the `std_scores` shows us how much the accuracy varies between folds. A smaller standard deviation means the model's performance is more consistent.

In [ ]:
single_search = GridSearchCV(estimator=pipe, param_grid=param_grid, cv=inner_cv)
single_search.fit(X, y)

mean_scores = single_search.cv_results_['mean_test_score']
std_scores = single_search.cv_results_['std_test_score']
c_values = [param for param in single_search.cv_results_['param_svc__C']]

plt.figure(figsize=(6,4))
plt.errorbar(c_values, mean_scores, yerr=std_scores, fmt='-o', capsize=5)
plt.xscale('log')
plt.xlabel('C value (log scale)')
plt.ylabel('Mean CV Accuracy')
plt.title('Cross validation accuracy for different C values')
plt.show()

The circles on the plot show the average accuracy for each value of C. The vertical lines (called error bars) give you an idea of how much the accuracy changed across the different folds during cross-validation. If the error bars are quite tall at a certain C, it means the model’s performance was less consistent for that setting. On the other hand, shorter error bars mean the model was more stable.

> If the error bars are very large, it usually means the model's performance jumped around quite a bit between folds. In that case, it might be worth gathering more data or picking a simpler model that behaves more predictably.

### 4.2 Repeat the Tuning Multiple Times with Different Splits
Next, we run the same hyperparameter search several times, each time changing the random seed so the data gets shuffled and split in a slightly different way. This helps us see whether the best value for C stays consistent, no matter how the data is divided.

In [ ]:
repeat_runs = 10 # to change the random_state in the loop
best_cs = []

for seed in range(repeat_runs):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    gs = GridSearchCV(estimator=pipe, param_grid=param_grid, cv=cv)
    gs.fit(X, y)
    best_cs.append(gs.best_params_['svc__C'])

print(f"Ran {repeat_runs} rounds of 5-fold CV. Best C in each run: {best_cs}")

In [ ]:
repeat_runs = 10 # to change the random_state in the loop
best_cs = []

for seed in range(repeat_runs):
  print(seed)

We can tally how often each C was selected as best:

In [ ]:
best_cs_series = pd.Series(best_cs, dtype=float)
counts = best_cs_series.value_counts().sort_index()
print("Frequency of each C value being selected as best:")
for c_val, count in counts.items():
    print(f"C={c_val}: {count} times")

print()

plt.figure(figsize=(6,4))
sns.barplot(x=counts.index.astype(str), y=counts.values)
plt.xlabel('C value')
plt.ylabel(f'Count (out of {repeat_runs} runs)')
plt.title('Stability of selected C across multiple runs')
plt.show()

- If you keep seeing the same C value pop up in most of your runs, that’s a good sign your tuning process is quite stable for this parameter.  
- If a few different C values show up often, it means small changes in how the data is split are leading to different results. In that case, those values are probably performing pretty similarly in practice.

### 4.3. Practical Decision Making

If your stability chart shows multiple parameters tying for first place, how do you choose?

1. **The Tie-Breaker:** Always choose the simpler model. (e.g., A smaller `C` value means stronger regularisation and less overfitting).
2. **Still Unsure?:** Gather more data. A larger dataset usually clarifies which parameter truly performs best.
3. **Accuracy Isn't Everything:** If parameters are tied on Accuracy, check how they perform on **Precision**, **Recall**, or **F1-Score** to break the tie based on your specific business need.

---
## 5.&nbsp;Tips and Common Pitfalls 💡
To round things off, here are a few helpful tips and things to watch out for when working with model validation and hyperparameter tuning:

* **Beware Data Leakage:** Any preprocessing (scaling, imputing) MUST happen *inside* the cross-validation loops using Scikit-Learn `Pipelines`.
* **Use Stratified Sampling:** Always use `StratifiedKFold` for classification to ensure every fold has the same ratio of target classes.
* **Don't Panic Over Slight Instability:** Small variations in "best" parameters are normal. Default to the simplest model.
* **The Final Step:** Always retrain your final model on **100% of your data** before deployment!

---
## 6.&nbsp;Challenges 🙂
1. **Try out a different model**  
Swap out `SVC` for something else, like a `DecisionTreeClassifier` or `KNeighborsClassifier`. Use nested cross-validation to tune key settings, such as the depth of the Decision Tree or the value of *k* for KNN.

2. **Test things on a new dataset**  
Give another dataset a go – the Wine dataset is a good one to start with, or pick one that interests you. Use the same nested cross-validation and check whether the stability of the best parameters changes.

3. **Tweak more hyperparameters**  
In the earlier example, we only adjusted `C`. This time, you could also tune other `SVC` options, like the `kernel` or `gamma` (if you're using an RBF kernel). See how this makes the search more complex and what impact it has on the results.

4. **Try skipping nested CV (just to see what happens)**  
Run a simple `GridSearchCV` without an outer loop and compare the results to those from nested CV. It's a helpful way to understand how much difference a proper validation setup can make.

Enjoy exploring, and always remember: validating your models well helps make sure your results are solid and reliable!

#### &nbsp;Example Dataset and Setup 🛠️
In this challenge, we'll work with the classic [Iris flower dataset]('https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html'). It’s a well-known example in the world of machine learning and perfect for getting to grips with how model validation works.

**A quick look at the dataset**:
The Iris dataset includes measurements of different parts of iris flowers – sepal length, sepal width, petal length, and petal width. Alongside these features, each flower is labelled with its species: setosa, versicolor, or virginica. Since it’s a small and balanced dataset, it’s just right for learning and experimenting without things getting too overwhelming.

In [ ]:
from sklearn.datasets import load_iris

# Load the iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Create a DataFrame for easy exploration
df = pd.DataFrame(X, columns=iris.feature_names)
df['species'] = pd.Categorical.from_codes(y, iris.target_names)

# Quick peek at the data
df.head()

In [ ]:
# your code here